# Complete Pairwise Tanimoto Fold-Distance Tables — Hi Tasks

This notebook computes fold-to-fold structural distances for the Hi datasets used in the OOD-holdout vs random-shuffle analysis: DRD2, HIV and Sol.

The goal is to test whether the Lo-Hi structural split is also reflected in the activity-relevant feature subspace.

For each dataset and each fold pair, we compute distances in:

- the full ECFP4 space;
- global activity top-k ECFP4 bits;
- OOD-selected activity top-k ECFP4 bits;
- random-shuffle-selected activity top-k ECFP4 bits;
- random top-k ECFP4 bits as a dimensionality-control baseline.

The main distance is the complete cross-fold pairwise Tanimoto distance:

\[
D(A,B)=\frac{1}{|A||B|}\sum_{x\in A}\sum_{y\in B} \left(1-T(x,y)\right)
\]

where \(T(x,y)\) is the Tanimoto similarity between two ECFP4 fingerprints.

Nearest-neighbour distances are saved only as a secondary diagnostic. Restricted-space results must be interpreted together with the valid molecule fraction, because at small k some molecules may have all-zero restricted fingerprints.

Outputs are saved to:

`results/results_fold_distance_tanimoto/hi/`

In [1]:
import json
import sys
import time
import warnings
import zlib
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    from scipy.stats import wasserstein_distance_nd

    _HAS_WASSERSTEIN = True
except Exception:
    _HAS_WASSERSTEIN = False


def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd().resolve()
    current = start
    while current != current.parent:
        if all((current / d).exists() for d in ["data", "utils", "training"]):
            return current
        current = current.parent
    raise RuntimeError(
        "Could not find project root containing data/, utils/, training/."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.fingerprints import compute_fingerprints

print(f"Project root: {PROJECT_ROOT}")
print(f"scipy wasserstein_distance_nd available: {_HAS_WASSERSTEIN}")

Project root: /home/f.capria/drug-discovery-lohi
scipy wasserstein_distance_nd available: True


In [ ]:
TASK = "hi"
DATASETS_MAIN = ["drd2", "hiv", "sol"]
DATASETS = DATASETS_MAIN

DATASET_LABELS = {"drd2": "DRD2", "hiv": "HIV", "sol": "Sol"}

SUBSET_FILES = {"F1": "test_3.csv", "F2": "test_2.csv", "F3": "test_1.csv"}
PAIRS = list(combinations(["F1", "F2", "F3"], 2))

PAIR_TO_OUTER_FOLD = {"F1_vs_F2": 1, "F1_vs_F3": 2, "F2_vs_F3": 3}

FP_TYPE = "ecfp4"
EXPECTED_ECFP4_BITS = 2048

K_VALUES = [10, 20, 50, 100, 200]
PAIRWISE_CHUNK_SIZE = 512
N_RANDOM_BIT_REPEATS = 30

VERBOSE_PAIRWISE = False
PRINT_EVERY_RANDOM_REPEAT = 5

RUN_WASSERSTEIN = False
W_SUBSAMPLE = 400

RANDOM_STATE = 42

DATA_ROOT = PROJECT_ROOT / "data" / TASK
OOD_CROSS_DIR = (
    PROJECT_ROOT / "results" / "results_ood_vs_random_shuffle" / TASK / "cross_dataset"
)
OUT_ROOT = PROJECT_ROOT / "results" / "results_fold_distance_tanimoto" / TASK
FIG_ROOT = OUT_ROOT / "figures"
CACHE_ROOT = PROJECT_ROOT / "features" / "fold_distance_tanimoto" / TASK

for d in (OUT_ROOT, FIG_ROOT, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)

print(f"OUT_ROOT: {OUT_ROOT}")
print(f"FIG_ROOT: {FIG_ROOT}")
print(f"CACHE_ROOT: {CACHE_ROOT}")

OUT_ROOT: /home/f.capria/drug-discovery-lohi/results/results_fold_distance_tanimoto/hi
FIG_ROOT: /home/f.capria/drug-discovery-lohi/results/results_fold_distance_tanimoto/hi/figures
CACHE_ROOT: /home/f.capria/drug-discovery-lohi/features/fold_distance_tanimoto/hi


In [3]:
MODEL_NAME_MAP = {
    "Decision Tree": "DT",
    "Logistic Regression": "LR",
    "Linear SVM": "SVM",
    "dt": "DT",
    "decision_tree": "DT",
    "lr": "LR",
    "logistic_regression": "LR",
    "svm": "SVM",
    "svm_linear": "SVM",
    "DT": "DT",
    "LR": "LR",
    "SVM": "SVM",
}

FP_MAP = {
    "ECFP4": "ecfp4",
    "MACCS": "maccs",
    "RDKit desc": "rdkit_desc",
    "ecfp4": "ecfp4",
    "maccs": "maccs",
    "rdkit_desc": "rdkit_desc",
}

IMPORTANCE_COL_CANDIDATES = [
    "importance_value",
    "permutation_importance_mean",
    "abs_weight",
    "normalized_abs_importance",
    "tree_importance",
]

In [4]:
def stable_seed(*parts, base: int = RANDOM_STATE) -> int:
    """Deterministic, cross-platform seed from arbitrary tuple of parts."""
    s = "|".join(str(p) for p in parts) + f"|{base}"
    return zlib.crc32(s.encode("utf-8")) & 0xFFFFFFFF


def local_rng(*parts) -> np.random.Generator:
    return np.random.default_rng(stable_seed(*parts))


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def get_smiles_col(df: pd.DataFrame) -> str:
    for col in ["smiles", "SMILES", "canonical_smiles"]:
        if col in df.columns:
            return col
    raise ValueError(f"No SMILES column found. Columns: {df.columns.tolist()}")


def find_importance_col(df: pd.DataFrame) -> str:
    if "importance_value" in df.columns:
        return "importance_value"
    for c in IMPORTANCE_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(
        f"No importance column found among {IMPORTANCE_COL_CANDIDATES}. "
        f"Columns: {df.columns.tolist()}"
    )


def normalize_model_name(x) -> str:
    s = str(x).strip()
    return MODEL_NAME_MAP.get(s, s)


def normalize_fingerprint_name(x) -> str:
    s = str(x).strip()
    return FP_MAP.get(s, s)


def protocol_match(x) -> str:
    s = str(x).strip().lower()
    if s in {"ood holdout", "ood_holdout", "holdout_ood", "ood"}:
        return "ood"
    if s in {"random shuffle", "random_shuffle", "random"}:
        return "random"
    if "ood" in s and "random" not in s:
        return "ood"
    if "random" in s:
        return "random"
    return s

In [ ]:
def load_subset_fps(dataset: str) -> dict[str, tuple[np.ndarray, pd.DataFrame]]:
    """Read F1/F2/F3 SMILES, compute ECFP4 with cache, drop invalid SMILES."""
    print_section(f"Loading subsets: {dataset.upper()}-{TASK}")

    data_dir = DATA_ROOT / dataset
    cache_dir = CACHE_ROOT / dataset
    cache_dir.mkdir(parents=True, exist_ok=True)

    out = {}
    for subset, fname in SUBSET_FILES.items():
        df = pd.read_csv(data_dir / fname).copy()
        smi_col = get_smiles_col(df)
        smiles = df[smi_col].astype(str).tolist()
        raw_n = len(smiles)

        t0 = time.time()
        X, valid_mask = compute_fingerprints(
            smiles_list=smiles,
            fp_type=FP_TYPE,
            cache_path=str(cache_dir / f"{subset}_{FP_TYPE}.npz"),
        )
        elapsed = time.time() - t0

        valid_mask = np.asarray(valid_mask, dtype=bool)
        if len(valid_mask) != raw_n:
            raise ValueError(
                f"valid_mask length mismatch: {len(valid_mask)} vs {raw_n}"
            )
        df_valid = df.loc[valid_mask].reset_index(drop=True)

        X = np.asarray(X, dtype=np.uint8)
        n_valid = X.shape[0]
        if n_valid != len(df_valid):
            raise ValueError(
                f"X/df mismatch after valid_mask: {n_valid} vs {len(df_valid)}"
            )
        if X.shape[1] != EXPECTED_ECFP4_BITS:
            raise ValueError(
                f"Expected {EXPECTED_ECFP4_BITS} bits, got {X.shape[1]} for {subset}"
            )

        invalid_n = raw_n - n_valid
        print(
            f"  {subset} ({fname}): raw={raw_n}, valid={n_valid}, "
            f"invalid={invalid_n}, n_bits={X.shape[1]}, time={elapsed:.1f}s"
        )

        out[subset] = (X, df_valid)

    return out


print("Fingerprint loading function defined.")

## Distance computation choices

- **Full ECFP4** is the baseline structural distance and the reference for everything else.
- **Activity-restricted spaces** (`global_topk`, `ood_topk`, `random_topk`) test whether the shift lies in features that the activity model actually relies on.
- **Random top-k bits** is the negative control: if activity restriction gives the same distance as random restriction of the same dimensionality, the apparent shift in the restricted space is just a dimensionality artefact, not evidence of activity-relevance.
- **Valid-molecule fractions** are required because at small k some molecules may have all-zero restricted fingerprints; restricted-space distances must be interpreted in light of how many molecules actually contribute.


In [ ]:
def nn_max_sim(X: np.ndarray, Y: np.ndarray, chunk: int = 512) -> np.ndarray:
    """For each row in X, return max Tanimoto similarity to any row in Y."""
    X = X.astype(np.float32, copy=False)
    Y = Y.astype(np.float32, copy=False)
    sx = X.sum(axis=1)
    sy = Y.sum(axis=1)
    n = X.shape[0]
    out = np.zeros(n, dtype=np.float32)
    for i in range(0, n, chunk):  # chunk for the dimensionality
        block = X[i : i + chunk]
        sb = sx[i : i + chunk]
        dots = block @ Y.T  # intersection
        denom = sb[:, None] + sy[None, :] - dots  # union
        with np.errstate(divide="ignore", invalid="ignore"):
            sim = np.where(denom > 0, dots / denom, 0.0)
        out[i : i + chunk] = sim.max(axis=1)
    return out


def pair_sim(
    X: np.ndarray, Y: np.ndarray, idx_x: np.ndarray, idx_y: np.ndarray
) -> np.ndarray:
    """Tanimoto similarity over a set of paired rows. Returns NaN where union is zero."""
    xa = X[idx_x].astype(np.float32, copy=False)
    yb = Y[idx_y].astype(np.float32, copy=False)
    dots = (xa * yb).sum(axis=1)
    sa = xa.sum(axis=1)
    sb = yb.sum(axis=1)
    denom = sa + sb - dots
    with np.errstate(divide="ignore", invalid="ignore"):
        sim = np.where(denom > 0, dots / denom, np.nan)
    return sim


def nn_distances(XA: np.ndarray, XB: np.ndarray) -> dict:
    """Symmetric nearest-neighbour Tanimoto distance, restricted-space aware."""
    sa = XA.sum(axis=1)
    sb = XB.sum(axis=1)
    valid_a = sa > 0
    valid_b = sb > 0
    n_total_a = int(XA.shape[0])
    n_total_b = int(XB.shape[0])

    XA_v = XA[valid_a]
    XB_v = XB[valid_b]

    if XA_v.shape[0] == 0 or XB_v.shape[0] == 0:
        return {
            "nn_sym_distance": np.nan,
            "nn_A_to_B_mean": np.nan,
            "nn_B_to_A_mean": np.nan,
            "nn_AB_array": np.array([], dtype=np.float32),
            "nn_BA_array": np.array([], dtype=np.float32),
            "n_valid_a": int(valid_a.sum()),
            "n_valid_b": int(valid_b.sum()),
            "n_total_a": n_total_a,
            "n_total_b": n_total_b,
        }

    # to maintain simmetry
    sim_AB = nn_max_sim(XA_v, XB_v)
    sim_BA = nn_max_sim(XB_v, XA_v)
    d_AB = 1.0 - sim_AB
    d_BA = 1.0 - sim_BA

    return {
        "nn_sym_distance": float(
            0.5 * (d_AB.mean() + d_BA.mean())
        ),  # mean for simmetry
        "nn_A_to_B_mean": float(d_AB.mean()),
        "nn_B_to_A_mean": float(d_BA.mean()),
        "nn_AB_array": d_AB,
        "nn_BA_array": d_BA,
        "n_valid_a": int(valid_a.sum()),
        "n_valid_b": int(valid_b.sum()),
        "n_total_a": n_total_a,
        "n_total_b": n_total_b,
    }


def complete_pairwise_distance(
    XA: np.ndarray,
    XB: np.ndarray,
    dataset: str,
    pair: str,
    space: str,
    chunk: int = PAIRWISE_CHUNK_SIZE,
) -> dict:
    """
    Complete cross-fold pairwise Tanimoto distance.

    Computes the mean Tanimoto distance over all pairs
    """
    XA = XA.astype(np.float32, copy=False)
    XB = XB.astype(np.float32, copy=False)

    sum_dist = 0.0
    n_valid = 0
    n_total = int(XA.shape[0] * XB.shape[0])
    if VERBOSE_PAIRWISE:
        print(
            f"      pairwise {dataset} | {pair} | {space}: "
            f"{XA.shape[0]} x {XB.shape[0]} = {n_total:,} pairs"
        )

    y_sum = XB.sum(axis=1)

    for start in range(0, XA.shape[0], chunk):
        xb = XA[start : start + chunk]
        x_sum = xb.sum(axis=1)

        inter = xb @ XB.T
        union = x_sum[:, None] + y_sum[None, :] - inter

        valid = union > 0

        sim = np.zeros_like(inter, dtype=np.float32)
        np.divide(inter, union, out=sim, where=valid)

        dist = 1.0 - sim

        sum_dist += float(dist[valid].sum())
        n_valid += int(valid.sum())

    mean_dist = sum_dist / n_valid if n_valid > 0 else np.nan

    if VERBOSE_PAIRWISE:
        print(
            f"        done: mean={mean_dist:.4f}, "
            f"valid_pairs={n_valid:,}/{n_total:,} "
            f"({n_valid / n_total:.3f})"
        )

    return {
        "pairwise_distance": mean_dist,
        "n_valid_pairs": n_valid,
        "n_total_pairs": n_total,
        "valid_pair_fraction": n_valid / n_total if n_total > 0 else np.nan,
        "pairwise_mode": "complete",
    }

## Load list A --> activity relevant features


In [ ]:
def load_list_a() -> pd.DataFrame:
    """Load List A activity feature importance, normalised and ECFP4-only."""
    print_section("Loading List A")

    path = OOD_CROSS_DIR / "cross_dataset_feature_importance_all.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing List A file: {path}")

    df = pd.read_csv(path, low_memory=False).copy()
    print(f"Raw List A: {df.shape}")

    df["dataset"] = df["dataset"].astype(str).str.lower()
    df = df[df["dataset"].isin(DATASETS_MAIN)].copy()

    df["model"] = df["model"].map(lambda x: normalize_model_name(x)).astype(str)
    df["fingerprint"] = (
        df["fingerprint"].map(lambda x: normalize_fingerprint_name(x)).astype(str)
    )
    df = df[df["fingerprint"] == "ecfp4"].copy()

    imp_col = find_importance_col(df)
    print(f"Using importance column: {imp_col}")

    df["importance_value_numeric"] = df[imp_col].fillna(0).astype(float)

    df["abs_importance"] = np.where(
        df["model"].eq("DT"),
        df["importance_value_numeric"].clip(lower=0.0),
        df["importance_value_numeric"].abs(),
    )

    df["feature_idx"] = df["feature_idx"].astype(int)
    df["fold"] = df["fold"].astype(int)
    df["protocol_norm"] = df["protocol"].map(protocol_match)

    print(f"Filtered List A: {df.shape}")
    print(f"Datasets : {sorted(df['dataset'].unique())}")
    print(f"Models   : {sorted(df['model'].unique())}")
    print(f"Protocols: {sorted(df['protocol_norm'].unique())}")
    print(f"Folds    : {sorted(df['fold'].unique())}")
    print("Rows by dataset/model:")
    print(
        df.groupby(["dataset", "model"])
        .size()
        .rename("n_rows")
        .reset_index()
        .to_string(index=False)
    )

    return df


list_a = load_list_a()

print("\nList A loaded.")
print(f"Shape: {list_a.shape}")
display(list_a.head())

## Best models


In [ ]:
def best_ecfp4_activity_models() -> dict[str, str]:
    """Choose one best ECFP4 activity model per dataset by mean final test PR-AUC."""
    print_section("Selecting best ECFP4 activity model per dataset")

    path = OOD_CROSS_DIR / "cross_dataset_protocol_summary.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing protocol summary: {path}")

    df = pd.read_csv(path).copy()
    df["dataset"] = df["dataset"].astype(str).str.lower()
    df = df[df["dataset"].isin(DATASETS_MAIN)].copy()
    df["model"] = df["model"].map(normalize_model_name).astype(str)
    df["fingerprint"] = df["fingerprint"].map(normalize_fingerprint_name).astype(str)
    df = df[df["fingerprint"] == "ecfp4"].copy()

    if "test_mean" in df.columns:
        score_col = "test_mean"
    elif "test_pr_auc_mean" in df.columns:
        score_col = "test_pr_auc_mean"
    else:
        raise ValueError(
            "No score column found among {test_mean, test_pr_auc_mean}. "
            f"Columns: {df.columns.tolist()}"
        )
    print(f"Using score column: {score_col}")

    agg = (
        df.groupby(["dataset", "model"], as_index=False)[score_col]
        .mean()
        .sort_values(["dataset", score_col], ascending=[True, False])
    )
    print("\nMean ECFP4 test score by dataset/model:")
    print(agg.to_string(index=False))

    best = (
        agg.sort_values([score_col], ascending=False)
        .groupby("dataset")
        .head(1)
        .set_index("dataset")["model"]
        .to_dict()
    )
    print("\nBest ECFP4 model per dataset:")
    for d, m in best.items():
        print(f"  {d}: {m}")

    return best


best_models = best_ecfp4_activity_models()

print("\nBest models selected:")
for ds, model in best_models.items():
    print(f"  {ds}: {model}")

In [9]:
def topk_bits_global(
    list_a: pd.DataFrame, dataset: str, model: str, k: int
) -> np.ndarray:
    """Top-k ECFP4 bits pooled across protocols and folds for one (dataset, model)."""
    sub = list_a[(list_a["dataset"] == dataset) & (list_a["model"] == model)]
    if sub.empty:
        return np.array([], dtype=int)
    agg = (
        sub.groupby("feature_idx")["abs_importance"].mean().sort_values(ascending=False)
    )
    return agg.head(k).index.to_numpy(dtype=int)


def topk_bits_fold_protocol(
    list_a: pd.DataFrame,
    dataset: str,
    model: str,
    fold: int,
    protocol: str,
    k: int,
) -> np.ndarray:
    """Top-k ECFP4 bits for (dataset, model, fold, protocol) — fold/protocol-aware."""
    protocol = protocol_match(protocol)
    sub = list_a[
        (list_a["dataset"] == dataset)
        & (list_a["model"] == model)
        & (list_a["fold"].astype(int) == int(fold))
        & (list_a["protocol_norm"] == protocol)
    ]
    if sub.empty:
        return np.array([], dtype=int)
    agg = (
        sub.groupby("feature_idx")["abs_importance"].mean().sort_values(ascending=False)
    )
    return agg.head(k).index.to_numpy(dtype=int)


def random_topk_bits(dataset: str, pair: str, k: int, repeat: int) -> np.ndarray:
    """Random subset of k ECFP4 bits, deterministic by (dataset, pair, k, repeat)."""
    rng = local_rng("random_bits", dataset, pair, k, repeat)
    return rng.choice(EXPECTED_ECFP4_BITS, size=k, replace=False)

In [ ]:
def restrict_to_bits(X: np.ndarray, bits: np.ndarray) -> np.ndarray:
    if len(bits) == 0:
        return np.zeros((X.shape[0], 0), dtype=X.dtype)
    return X[:, bits]


def wasserstein_nd_optional(
    XA: np.ndarray,
    XB: np.ndarray,
    n: int,
    dataset: str,
    pair: str,
    space: str,
) -> float:
    """Optional ND Wasserstein-1 distance with Euclidean ground metric. Disabled by default."""
    if not RUN_WASSERSTEIN or not _HAS_WASSERSTEIN:
        return np.nan
    rng = local_rng("wasserstein", dataset, pair, space)
    if XA.shape[0] > n:
        XA = XA[rng.choice(XA.shape[0], size=n, replace=False)]
    if XB.shape[0] > n:
        XB = XB[rng.choice(XB.shape[0], size=n, replace=False)]
    try:
        return float(
            wasserstein_distance_nd(XA.astype(np.float32), XB.astype(np.float32))
        )
    except Exception as exc:
        warnings.warn(f"Wasserstein failed for {dataset}|{pair}|{space}: {exc}")
        return np.nan


print("Top-k bit selection and optional Wasserstein functions defined.")

In [ ]:
DIST_ROW_KEYS = [
    "dataset",
    "dataset_label",
    "pair",
    "space",
    "k",
    "model",
    "bit_source",
    "activity_protocol",
    "activity_fold",
    "bit_repeat",
    "bits_used",
    "nn_sym_distance",
    "nn_A_to_B_mean",
    "nn_B_to_A_mean",
    "wasserstein_nd",
    "valid_molecule_fraction",
    "n_valid_molecules",
    "n_total_molecules",
    "pairwise_distance",
    "valid_pair_fraction",
    "n_valid_pairs",
    "n_total_pairs",
    "pairwise_mode",
]


def _base_row(
    dataset,
    pair,
    space,
    k,
    model,
    bit_source,
    activity_protocol,
    activity_fold,
    bit_repeat,
    bits_used,
):
    return {
        "dataset": dataset,
        "dataset_label": DATASET_LABELS.get(dataset, dataset),
        "pair": pair,
        "space": space,
        "k": int(k),
        "model": model,
        "bit_source": bit_source,
        "activity_protocol": activity_protocol,
        "activity_fold": activity_fold,
        "bit_repeat": bit_repeat,
        "bits_used": int(bits_used),
    }


def add_distance_row(
    dist_rows,
    hist_rows,
    dataset,
    pair,
    XA,
    XB,
    model,
):
    """Full ECFP4 space row."""
    space = "full_ecfp4"
    bit_source = "full_ecfp4"
    nn = nn_distances(XA, XB)
    pw = complete_pairwise_distance(
        XA,
        XB,
        dataset=dataset,
        pair=pair,
        space=space,
    )
    wd = wasserstein_nd_optional(XA, XB, W_SUBSAMPLE, dataset, pair, space)

    n_valid_mol = nn["n_valid_a"] + nn["n_valid_b"]
    n_total_mol = nn["n_total_a"] + nn["n_total_b"]

    row = _base_row(
        dataset,
        pair,
        space,
        EXPECTED_ECFP4_BITS,
        model,
        bit_source,
        np.nan,
        np.nan,
        np.nan,
        EXPECTED_ECFP4_BITS,
    )
    row.update(
        {
            "nn_sym_distance": nn["nn_sym_distance"],
            "nn_A_to_B_mean": nn["nn_A_to_B_mean"],
            "nn_B_to_A_mean": nn["nn_B_to_A_mean"],
            "wasserstein_nd": wd,
            "valid_molecule_fraction": n_valid_mol / max(n_total_mol, 1),
            "n_valid_molecules": n_valid_mol,
            "n_total_molecules": n_total_mol,
            "pairwise_distance": pw["pairwise_distance"],
            "valid_pair_fraction": pw["valid_pair_fraction"],
            "n_valid_pairs": pw["n_valid_pairs"],
            "n_total_pairs": pw["n_total_pairs"],
            "pairwise_mode": pw["pairwise_mode"],
        }
    )
    dist_rows.append(row)

    for d in nn["nn_AB_array"]:
        hist_rows.append(
            {
                "dataset": dataset,
                "pair": pair,
                "space": space,
                "k": EXPECTED_ECFP4_BITS,
                "bit_source": bit_source,
                "direction": "A_to_B",
                "nn_distance": float(d),
            }
        )
    for d in nn["nn_BA_array"]:
        hist_rows.append(
            {
                "dataset": dataset,
                "pair": pair,
                "space": space,
                "k": EXPECTED_ECFP4_BITS,
                "bit_source": bit_source,
                "direction": "B_to_A",
                "nn_distance": float(d),
            }
        )


def add_restricted_distance_row(
    dist_rows,
    hist_rows,
    dataset,
    pair,
    XA,
    XB,
    bits,
    k,
    model,
    space,
    bit_source,
    activity_protocol,
    activity_fold,
    bit_repeat,
    store_hist: bool,
):
    """Top-k restricted space row."""
    if len(bits) == 0 and bit_source in {
        "global_activity",
        "ood_activity",
        "random_shuffle_activity",
    }:
        warnings.warn(
            f"No activity bits found for {dataset} | {pair} | {space} | "
            f"model={model} | bit_source={bit_source}. "
            "Distances will be NaN or based on empty restricted fingerprints."
        )
    XA_r = restrict_to_bits(XA, bits)
    XB_r = restrict_to_bits(XB, bits)

    nn = nn_distances(XA_r, XB_r)
    pw = complete_pairwise_distance(
        XA_r,
        XB_r,
        dataset=dataset,
        pair=pair,
        space=space,
    )
    wd = wasserstein_nd_optional(XA_r, XB_r, W_SUBSAMPLE, dataset, pair, space)

    n_valid_mol = nn["n_valid_a"] + nn["n_valid_b"]
    n_total_mol = nn["n_total_a"] + nn["n_total_b"]

    row = _base_row(
        dataset,
        pair,
        space,
        k,
        model,
        bit_source,
        activity_protocol,
        activity_fold,
        bit_repeat,
        len(bits),
    )
    row.update(
        {
            "nn_sym_distance": nn["nn_sym_distance"],
            "nn_A_to_B_mean": nn["nn_A_to_B_mean"],
            "nn_B_to_A_mean": nn["nn_B_to_A_mean"],
            "wasserstein_nd": wd,
            "valid_molecule_fraction": n_valid_mol / max(n_total_mol, 1),
            "n_valid_molecules": n_valid_mol,
            "n_total_molecules": n_total_mol,
            "pairwise_distance": pw["pairwise_distance"],
            "valid_pair_fraction": pw["valid_pair_fraction"],
            "n_valid_pairs": pw["n_valid_pairs"],
            "n_total_pairs": pw["n_total_pairs"],
            "pairwise_mode": pw["pairwise_mode"],
        }
    )
    dist_rows.append(row)

    if store_hist:
        for d in nn["nn_AB_array"]:
            hist_rows.append(
                {
                    "dataset": dataset,
                    "pair": pair,
                    "space": space,
                    "k": k,
                    "bit_source": bit_source,
                    "direction": "A_to_B",
                    "nn_distance": float(d),
                }
            )
        for d in nn["nn_BA_array"]:
            hist_rows.append(
                {
                    "dataset": dataset,
                    "pair": pair,
                    "space": space,
                    "k": k,
                    "bit_source": bit_source,
                    "direction": "B_to_A",
                    "nn_distance": float(d),
                }
            )


print("Distance row-builder functions defined.")

### Compute dataset distances


In [ ]:
def compute_dataset_distances(
    dataset: str,
    fps: dict[str, tuple[np.ndarray, pd.DataFrame]],
    list_a: pd.DataFrame,
    best_model: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    print_section(f"Computing distances: {dataset.upper()}-{TASK} (model={best_model})")
    dist_rows: list[dict] = []
    hist_rows: list[dict] = []

    for fold_a, fold_b in PAIRS:
        pair = f"{fold_a}_vs_{fold_b}"
        outer_fold = PAIR_TO_OUTER_FOLD[pair]
        XA, _ = fps[fold_a]
        XB, _ = fps[fold_b]
        print(
            f"\n  Pair {pair} (outer fold {outer_fold}): nA={XA.shape[0]} nB={XB.shape[0]}"
        )

        #  Full ECFP4
        add_distance_row(dist_rows, hist_rows, dataset, pair, XA, XB, best_model)

        # Restricted top-k spaces
        for k in K_VALUES:
            print(f"    k={k}: activity spaces + random-bit baseline")
            # Global activity bits (pooled across protocols and folds)
            bits_g = topk_bits_global(list_a, dataset, best_model, k)
            print(f"      global_activity: {len(bits_g)} bits")
            add_restricted_distance_row(
                dist_rows,
                hist_rows,
                dataset,
                pair,
                XA,
                XB,
                bits=bits_g,
                k=k,
                model=best_model,
                space=f"global_top{k}",
                bit_source="global_activity",
                activity_protocol="pooled",
                activity_fold=np.nan,
                bit_repeat=np.nan,
                store_hist=True,
            )

            # OOD activity bits, fold-aware
            bits_ood = topk_bits_fold_protocol(
                list_a, dataset, best_model, outer_fold, "ood", k
            )
            print(f"      ood_activity fold={outer_fold}: {len(bits_ood)} bits")
            add_restricted_distance_row(
                dist_rows,
                hist_rows,
                dataset,
                pair,
                XA,
                XB,
                bits=bits_ood,
                k=k,
                model=best_model,
                space=f"ood_top{k}",
                bit_source="ood_activity",
                activity_protocol="ood",
                activity_fold=outer_fold,
                bit_repeat=np.nan,
                store_hist=True,
            )

            # Random-shuffle activity bits, fold-aware
            bits_rnd = topk_bits_fold_protocol(
                list_a, dataset, best_model, outer_fold, "random", k
            )
            print(
                f"      random_shuffle_activity fold={outer_fold}: {len(bits_rnd)} bits"
            )
            add_restricted_distance_row(
                dist_rows,
                hist_rows,
                dataset,
                pair,
                XA,
                XB,
                bits=bits_rnd,
                k=k,
                model=best_model,
                space=f"random_top{k}",
                bit_source="random_shuffle_activity",
                activity_protocol="random",
                activity_fold=outer_fold,
                bit_repeat=np.nan,
                store_hist=True,
            )

            # Random-bit baseline (negative control for dimensionality)
            for r in range(N_RANDOM_BIT_REPEATS):
                if (
                    r == 0
                    or (r + 1) % PRINT_EVERY_RANDOM_REPEAT == 0
                    or r == N_RANDOM_BIT_REPEATS - 1
                ):
                    print(f"      random_bits repeat {r + 1}/{N_RANDOM_BIT_REPEATS}")
                bits_rb = random_topk_bits(dataset, pair, k, r)
                add_restricted_distance_row(
                    dist_rows,
                    hist_rows,
                    dataset,
                    pair,
                    XA,
                    XB,
                    bits=bits_rb,
                    k=k,
                    model=best_model,
                    space=f"random_bits_top{k}",
                    bit_source="random_bits",
                    activity_protocol=np.nan,
                    activity_fold=np.nan,
                    bit_repeat=r,
                    store_hist=False,
                )

    dist_df = pd.DataFrame(dist_rows)
    hist_df = pd.DataFrame(hist_rows)

    n_random_bit_rows = (dist_df["bit_source"] == "random_bits").sum()
    expected_rb = len(PAIRS) * len(K_VALUES) * N_RANDOM_BIT_REPEATS
    print(f"\n  Dataset {dataset}: dist_rows={len(dist_df)}, hist_rows={len(hist_df)}")
    print(
        f"  random_bits rows = {n_random_bit_rows} (expected {expected_rb}: "
        f"{len(PAIRS)} pairs × {len(K_VALUES)} k × {N_RANDOM_BIT_REPEATS} repeats)"
    )

    expected_total = len(PAIRS) * (
        1 + len(K_VALUES) * (1 + 1 + 1 + N_RANDOM_BIT_REPEATS)
    )
    if len(dist_df) != expected_total:
        warnings.warn(
            f"{dataset}: got {len(dist_df)} dist rows, expected {expected_total}"
        )

    print("Dataset-level distance computation function defined.")

    return dist_df, hist_df

### Compute distances dataset by dataset

In [ ]:
dist_parts = []
hist_parts = []

for ds in DATASETS:
    print_section(f"Running dataset: {ds.upper()}")

    if ds not in best_models:
        warnings.warn(f"No best ECFP4 model for {ds}; skipping.")
        continue

    print(f"Best model: {best_models[ds]}")

    fps = load_subset_fps(ds)

    t0 = time.time()

    dist_df, hist_df = compute_dataset_distances(
        dataset=ds,
        fps=fps,
        list_a=list_a,
        best_model=best_models[ds],
    )

    elapsed = time.time() - t0

    dist_parts.append(dist_df)
    hist_parts.append(hist_df)

    dataset_dist_path = OUT_ROOT / f"fold_distance_summary_{ds}.csv"
    dataset_hist_path = OUT_ROOT / f"fold_distance_nn_distributions_{ds}.csv"

    dist_df.to_csv(dataset_dist_path, index=False)
    hist_df.to_csv(dataset_hist_path, index=False)

    print(
        f"Saved {ds}: dist_rows={len(dist_df)}, hist_rows={len(hist_df)}, "
        f"time={elapsed / 60:.1f} min"
    )
    print(f"  {dataset_dist_path}")
    print(f"  {dataset_hist_path}")

### Concatenate all datasets and save global tables

In [ ]:
if not dist_parts:
    raise RuntimeError("No dataset produced distance rows.")

dist_all = pd.concat(dist_parts, ignore_index=True)
hist_all = pd.concat(hist_parts, ignore_index=True) if hist_parts else pd.DataFrame()

dist_path = OUT_ROOT / "fold_distance_summary.csv"
hist_path = OUT_ROOT / "fold_distance_nn_distributions.csv"

dist_all.to_csv(dist_path, index=False)
hist_all.to_csv(hist_path, index=False)

print("Saved global tables.")
print(f"dist_all: {dist_all.shape} -> {dist_path}")
print(f"hist_all: {hist_all.shape} -> {hist_path}")

display(dist_all.head())

In [20]:
def print_output_diagnostics(dist_all: pd.DataFrame, hist_all: pd.DataFrame):
    print_section("Output diagnostics")
    print(f"dist_all shape: {dist_all.shape}")
    print(f"hist_all shape: {hist_all.shape}")

    if len(dist_all) == 0:
        raise RuntimeError("dist_all is empty — distance computation produced no rows.")

    print("\nRows by dataset / bit_source:")
    print(
        dist_all.groupby(["dataset", "bit_source"])
        .size()
        .rename("n")
        .reset_index()
        .to_string(index=False)
    )

    expected_per_dataset = len(PAIRS) * (
        1 + len(K_VALUES) * (1 + 1 + 1 + N_RANDOM_BIT_REPEATS)
    )

    print("\nExpected row count per dataset:")
    for ds in DATASETS:
        got = int((dist_all["dataset"] == ds).sum())
        flag = "OK" if got == expected_per_dataset else "MISSING"
        print(f"  {ds}: got {got}, expected {expected_per_dataset} [{flag}]")

    required_spaces = ["full_ecfp4", "global_top50", "random_bits_top50"]
    present = set(dist_all["space"].unique())
    missing = [s for s in required_spaces if s not in present]

    if missing:
        warnings.warn(f"Missing required spaces: {missing}")
    else:
        print("\nRequired spaces present: " + ", ".join(required_spaces))

    if "pairwise_distance" not in dist_all.columns:
        raise RuntimeError("Missing main metric column: pairwise_distance")

    if dist_all["pairwise_distance"].notna().sum() == 0:
        raise RuntimeError("All pairwise_distance values are NaN — pipeline failed.")

    print(
        f"\nFinite pairwise_distance: "
        f"{dist_all['pairwise_distance'].notna().sum()}/{len(dist_all)}"
    )

    if "nn_sym_distance" in dist_all.columns:
        if dist_all["nn_sym_distance"].notna().sum() == 0:
            warnings.warn(
                "All nn_sym_distance values are NaN. "
                "Main pairwise distances are available, but NN diagnostic failed."
            )
        else:
            print(
                f"Finite nn_sym_distance: "
                f"{dist_all['nn_sym_distance'].notna().sum()}/{len(dist_all)}"
            )

    if "valid_pair_fraction" in dist_all.columns:
        print("\nValid pair fraction summary:")
        print(
            dist_all["valid_pair_fraction"]
            .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
            .to_string()
        )

    if "valid_molecule_fraction" in dist_all.columns:
        print("\nValid molecule fraction summary:")
        print(
            dist_all["valid_molecule_fraction"]
            .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
            .to_string()
        )

In [ ]:
def build_summary(dist_all: pd.DataFrame) -> pd.DataFrame:
    print_section("Building final summary")

    rows = []

    for k in [50, 100]:
        for ds in DATASETS:
            for fa, fb in PAIRS:
                pair = f"{fa}_vs_{fb}"
                base_q = (dist_all["dataset"] == ds) & (dist_all["pair"] == pair)

                full = dist_all[base_q & (dist_all["bit_source"] == "full_ecfp4")]

                act = dist_all[
                    base_q
                    & (dist_all["bit_source"] == "global_activity")
                    & (dist_all["k"] == k)
                ]

                rb = dist_all[
                    base_q
                    & (dist_all["bit_source"] == "random_bits")
                    & (dist_all["k"] == k)
                ]

                full_d = (
                    float(full["pairwise_distance"].mean()) if len(full) else np.nan
                )

                act_d = float(act["pairwise_distance"].mean()) if len(act) else np.nan

                rb_d_mean = float(rb["pairwise_distance"].mean()) if len(rb) else np.nan

                rb_q05 = (
                    float(rb["pairwise_distance"].quantile(0.05)) if len(rb) else np.nan
                )

                rb_q95 = (
                    float(rb["pairwise_distance"].quantile(0.95)) if len(rb) else np.nan
                )

                vm_frac = (
                    float(act["valid_molecule_fraction"].mean()) if len(act) else np.nan
                )

                vp_frac = (
                    float(act["valid_pair_fraction"].mean()) if len(act) else np.nan
                )

                full_nn = (
                    float(full["nn_sym_distance"].mean())
                    if len(full) and "nn_sym_distance" in full.columns
                    else np.nan
                )

                act_nn = (
                    float(act["nn_sym_distance"].mean())
                    if len(act) and "nn_sym_distance" in act.columns
                    else np.nan
                )

                rows.append(
                    {
                        "k": k,
                        "dataset": ds,
                        "dataset_label": DATASET_LABELS[ds],
                        "pair": pair,
                        # Main metric: complete cross-fold pairwise distance
                        "full_pairwise_distance_mean": full_d,
                        "activity_pairwise_distance_mean": act_d,
                        "delta_activity_minus_full_pairwise": (
                            act_d - full_d
                            if np.isfinite(act_d) and np.isfinite(full_d)
                            else np.nan
                        ),
                        "random_bits_pairwise_distance_mean": rb_d_mean,
                        "random_bits_pairwise_q05": rb_q05,
                        "random_bits_pairwise_q95": rb_q95,
                        "delta_activity_minus_random_bits_pairwise": (
                            act_d - rb_d_mean
                            if np.isfinite(act_d) and np.isfinite(rb_d_mean)
                            else np.nan
                        ),
                        # Coverage diagnostics
                        "valid_molecule_fraction_mean": vm_frac,
                        "valid_pair_fraction_mean": vp_frac,
                        # Secondary NN diagnostic
                        "full_nn_distance_mean": full_nn,
                        "activity_nn_distance_mean": act_nn,
                        "delta_activity_minus_full_nn": (
                            act_nn - full_nn
                            if np.isfinite(act_nn) and np.isfinite(full_nn)
                            else np.nan
                        ),
                    }
                )

    out = pd.DataFrame(rows)

    out_path = OUT_ROOT / "fold_distance_activity_vs_random_bits_summary.csv"
    out.to_csv(out_path, index=False)

    print(f"Saved summary: {out_path} ({len(out)} rows)")

    display_cols = [
        "k",
        "dataset",
        "pair",
        "full_pairwise_distance_mean",
        "activity_pairwise_distance_mean",
        "random_bits_pairwise_distance_mean",
        "delta_activity_minus_random_bits_pairwise",
        "valid_molecule_fraction_mean",
        "valid_pair_fraction_mean",
    ]

    print("\nSummary preview:")
    print(out[display_cols].head(12).to_string(index=False))

    return out


final_summary = build_summary(dist_all)

print("Final summary built.")
print(final_summary.shape)
display(final_summary)